In [12]:
library(stats)

df <- read.csv("data.csv")
head(df)

,name,yearEstablished,state,stateAbbreviation,locationType,locationTypeKind,typeOwned,typeProfit,level,hasUndergraduate,...,closedByRegion_2023,totalByState_2024,closedByState_2024,totalByRegion_2024,closedByRegion_2024,totalByState_2025,closedByState_2025,totalByRegion_2025,closedByRegion_2025,regionUnemploymentRate
,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,...,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>
1,Alderson Broaddus University,1871,West Virginia,WV,Rural,Distant,Private,Nonprofit,4-year,True,...,1,69,0,2055,4,69,0,2055,2,3.6
2,Alliance University (Formerly Nyack College),1882,New York,NY,City,Large,Private,Nonprofit,4-year,True,...,4,413,3,1163,8,413,0,1163,0,4.1
3,Ancilla College,1937,Indiana,IN,Rural,Distant,Private,Nonprofit,2-year,True,...,6,102,1,1323,9,102,0,1323,4,4.1
4,Becker College,1784,Massachusetts,MA,City,Midsize,Private,Nonprofit,4-year,True,...,4,144,1,1163,8,144,0,1163,0,4.1
5,Bloomfield College,1868,New Jersey,NJ,Suburb,Large,Private,Nonprofit,4-year,True,...,4,144,0,1163,8,144,0,1163,0,4.1
6,Bluffton University,1899,Ohio,OH,Town,Distant,Private,Nonprofit,4-year,True,...,6,274,1,1323,9,274,1,1323,4,4.1


In [2]:
# Data: number of closures (y) and total observations (n)
# Example: 50 closures out of 200 colleges observed
y <- 50  # Number of successes (closures)
n <- 200 # Total trials (colleges observed)
# Justify: y and n are observed counts, satisfying Binomial assumptions.

In [3]:
# Prior parameters: Beta(1, 1) for a uniform prior
a <- 1  # Shape parameter a
b <- 1  # Shape parameter b
# Justify: Beta(1, 1) is uniform over [0, 1], a non-informative prior, ensuring
# p(theta) = 1 for 0 <= theta <= 1, with E(theta) = a/(a + b) = 0.5.

In [4]:
# Posterior parameters: Beta(a + y, b + n - y)
a_post <- a + y  # a + K (successes)
b_post <- b + n - y  # b + N - K (failures)
# Justify: Conjugacy gives Beta(a + y, b + n - y). Your note’s Beta(y + 1, n - y + 1)
# holds if a = b = 1. Here, a_post = 51, b_post = 151, consistent with data update.

In [5]:
# Posterior mean
post_mean <- a_post / (a_post + b_post)
# Justify: E(theta|y) = (a + y)/(a + b + n) = 51/202 ≈ 0.252, approximating y/n = 50/200 = 0.25

In [6]:
# Posterior variance
post_var <- (a_post * b_post) / ((a_post + b_post)^2 * (a_post + b_post + 1))
# Justify: Var(theta|y) = ab/((a + b)^2 (a + b + 1)) for Beta(a, b). Here, ≈ 0.00093,
# smaller than prior Var(theta) = 1/12 ≈ 0.0833, per law of total variance.

In [7]:
# Predictive distribution for m = 100 future trials
m <- 100  # Number of future trials
pred_probs <- dbinom(0:m, size = m, prob = post_mean)
# Justify: Beta-Binomial E(\tilde{y}|y) = m * post_mean = 100 * 0.252 = 25.2,
# approximated by Bin(m, E(theta|y)).

In [8]:
# Output results
cat("Posterior: Beta(", a_post, ", ", b_post, ")\n")
cat("Posterior Mean: ", round(post_mean, 4), "\n")
cat("Posterior Variance: ", round(post_var, 4), "\n")
cat("Predictive Expectation (m = 100): ", round(m * post_mean, 1), "\n")

Posterior: Beta( 51 ,  151 )
Posterior Mean:  0.2525 
Posterior Variance:  9e-04 
Predictive Expectation (m = 100):  25.2 


In [ ]:
# Load library for Beta functions
library(stats)

# Read dataset
df <- read.csv("data.csv")
# Justify: Loads raw data with columns totalByState_[2020-2025] and closedByState_[2020-2025].

# Extract unique states and compute means
states <- unique(df$state)  # 28 unique states
n_s <- tapply(df$totalByStateMean, df$state, mean)  # Trials per state
y_s <- tapply(df$closedByStateMean, df$state, mean)  # Successes per state
# Justify: tapply aggregates by state. Means are used as per your design, but note that
# Binomial n and y should be integers. Means (e.g., 10.5 trials) are approximate counts.

# Initial prior: Beta(1, 1) for each state
a <- 1  # Shape parameter a
b <- 1  # Shape parameter b
# Justify: Uniform prior, E(theta) = 0.5, Var(theta) = 1/12 ≈ 0.0833, non-informative.

# Compute posterior parameters for each state
a_post <- a + y_s  # a + y_s
b_post <- b + n_s - y_s  # b + n_s - y_s
# Justify: Conjugate update. For Beta(a, b), posterior is Beta(a + y, b + n - y).
# Check: n_s - y_s must be positive; if negative, data is invalid (e.g., closures > total).

# Posterior means
post_means <- a_post / (a_post + b_post)
# Justify: E(theta_s|y_s) = (a + y_s)/(a + b + n_s). With a = b = 1, ≈ y_s/n_s for large n_s.

# Posterior variances
post_vars <- (a_post * b_post) / ((a_post + b_post)^2 * (a_post + b_post + 1))
# Justify: Var(theta_s|y_s) = ab/((a + b)^2 (a + b + 1)). Smaller than prior variance.

# Output results for each state
results <- data.frame(state = states, n_s = n_s, y_s = y_s,
                      a_post = a_post, b_post = b_post,
                      post_mean = post_means, post_var = post_vars)
print(results)
# Justify: Summarizes posteriors. Inspect for n_s < y_s (invalid) or NaNs.

# # Alternative prior: Beta(2, 2)
# a_alt <- 2
# b_alt <- 2
# a_post_alt <- a_alt + y_s
# b_post_alt <- b_alt + n_s - y_s
# post_means_alt <- a_post_alt / (a_post_alt + b_post_alt)
# results_alt <- data.frame(state = states, a_post = a_post_alt, b_post = b_post_alt, 
#                           post_mean = post_means_alt)
# print(results_alt)
# Justify: Beta(2, 2) has E(theta) = 1, Var(theta) = 1/20 = 0.05, slightly informative.

                 n_s  y_s a_post b_post   post_mean     post_var
Alabama        78.00 0.25   1.25  78.75 0.015625000 1.898872e-04
California    657.50 1.25   2.25 657.25 0.003411676 5.147670e-06
Delaware       16.00 0.25   1.25  16.75 0.069444444 3.401153e-03
Florida       320.25 0.00   1.00 321.25 0.003103181 9.570150e-06
Illinois      235.75 0.50   1.50 236.25 0.006309148 2.625903e-05
Indiana       102.00 0.25   1.25 102.75 0.012019231 1.130930e-04
Iowa           75.00 0.25   1.25  75.75 0.016233766 2.047466e-04
Maryland       77.00 0.00   1.00  78.00 0.012658228 1.562250e-04
Massachusetts 144.00 0.50   1.50 144.50 0.010273973 6.917291e-05
Michigan      154.00 0.50   1.50 154.50 0.009615385 6.065560e-05
Missouri      136.50 0.25   1.25 137.25 0.009025271 6.411337e-05
Nebraska       40.00 0.25   1.25  40.75 0.029761905 6.715380e-04
Nevada         35.00 0.25   1.25  35.75 0.033783784 8.590116e-04
New Hampshire  34.00 0.00   1.00  35.00 0.027777778 7.298966e-04
New Jersey    144.00 0.25